# 01 — Dataset Metadata Review

**Goal:** Understand what datasets exist in the pipeline before any dbt transformation work begins.

This notebook reviews the `audit` schema — specifically `metadata_datasets` and `metadata_columns` —
to answer:

- What datasets do we have, and where did they come from?
- What's their grain (country/year, country/crop/year, etc.)?
- What's their temporal and geographic coverage?
- Which datasets can be joined together?
- Are there missing or duplicate datasets?

Output of this notebook: a written **dataset understanding + integration plan**, saved at the bottom.

In [1]:
from _bootstrap import project_root
from src.audit.run_management import get_latest_run_id, list_run_ids

import polars as pl

# Show all columns and full-width string values in every cell below —
# otherwise polars truncates wide tables with "…" and cuts long strings like file_path/error
pl.Config.set_tbl_cols(-1)          # show all columns, no collapsing
pl.Config.set_tbl_width_chars(200)  # widen the rendered table
pl.Config.set_fmt_str_lengths(120)  # don't truncate long strings (e.g. file_path, error messages)
pl.Config.set_tbl_rows(50)          # show more rows before truncating vertically

# Connect to the project's DuckDB instance
from src.database.connection import get_duckdb_conn

conn = get_duckdb_conn(True)
print("Connected")

Connected


## 1. Confirm audit schema contents

Before querying, confirm which tables actually exist in `audit` and their exact schemas.
This avoids assuming column names that don't match reality.

In [2]:
# List all tables in the audit schema
tables = conn.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'audit'
    ORDER BY table_name;
""").fetchall()

print("Audit tables:\n")
for (table,) in tables:
    print(table)

Audit tables:

dq_reports
metadata_columns
metadata_datasets
profiling_reports


In [3]:
# Inspect exact column names/types for each audit table
# so downstream queries don't guess at field names
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    for t in ["metadata_datasets", "metadata_columns", "dq_reports", "profiling_reports"]:
        print(f"\n--- audit.{t} ---")
        schema = conn.sql(f"DESCRIBE audit.{t}").pl()
        print(schema)


--- audit.metadata_datasets ---
shape: (14, 6)
┌───────────────────┬─────────────┬──────┬──────┬───────────────────────────────────────────┬───────┐
│ column_name       ┆ column_type ┆ null ┆ key  ┆ default                                   ┆ extra │
│ ---               ┆ ---         ┆ ---  ┆ ---  ┆ ---                                       ┆ ---   │
│ str               ┆ str         ┆ str  ┆ str  ┆ str                                       ┆ str   │
╞═══════════════════╪═════════════╪══════╪══════╪═══════════════════════════════════════════╪═══════╡
│ metadata_id       ┆ BIGINT      ┆ NO   ┆ PRI  ┆ nextval('audit.metadata_datasets_id_seq') ┆ null  │
│ run_id            ┆ VARCHAR     ┆ NO   ┆ UNI  ┆ null                                      ┆ null  │
│ dataset_name      ┆ VARCHAR     ┆ NO   ┆ UNI  ┆ null                                      ┆ null  │
│ file_path         ┆ VARCHAR     ┆ YES  ┆ null ┆ null                                      ┆ null  │
│ file_format       ┆ VARCHAR     

### Notes — Audit Schema

- `metadata_datasets` — one row per dataset per run: path, format, size, row/column count, source, timestamps.
- `metadata_columns` — one row per column per dataset per run: dtype, nullability, null %, numeric/temporal/categorical flags, sample values (JSON).
- `dq_reports` — data quality pass/fail summary per table per run (feeds Notebook 02).
- `profiling_reports` — profiling summary per table per run (feeds Notebook 03).
- All four tables are `run_id`-keyed, i.e. **append-only logs**, not current-state tables. Every query must pin to a specific `run_id`.

## 2. Resolve the latest run_id

`metadata_datasets` accumulates new rows every time the metadata generator runs.
We use `get_latest_run_id()` / `list_run_ids()` from `src.utils.run_id` to pin this
notebook to one consistent snapshot instead of mixing rows across runs.

In [4]:
from src.audit.run_management import get_latest_run_id, list_run_ids

# Check what runs actually exist before picking one
all_runs = list_run_ids(conn, "audit.metadata_datasets")
print(f"Total runs recorded: {len(all_runs)}")
for r in all_runs:
    print(" ", r)

Total runs recorded: 2
  metadata_20260707_141136
  metadata_20260707_130152


In [5]:
# The run_id prefix used by this project's metadata generator is "metadata"
# (not "raw"/"staging"/"mart" as the utility docstring suggests)
LAYER = "metadata"

latest_run_id = get_latest_run_id(conn, "audit.metadata_datasets", layer=LAYER)
print(f"Using run_id: {latest_run_id}")

Using run_id: metadata_20260707_141136


In [6]:
# Check run_id prefixes used by dq_reports and profiling_reports separately —
# these may follow a different convention since they key off layer/table_name, not dataset_name
for t in ["audit.dq_reports", "audit.profiling_reports"]:
    runs = list_run_ids(conn, t)
    print(f"\n{t}: {len(runs)} runs")
    for r in runs[:5]:
        print(" ", r)


audit.dq_reports: 2 runs
  raw_2026-07-07T18:57:29
  raw_2026-07-07T18:08:39

audit.profiling_reports: 1 runs
  raw_2026-07-07T19:16:43


### Notes — Run Selection

- Runs found in `metadata_datasets`: `metadata_20260707_130152` (older), `metadata_20260707_141136` (newer).
- This notebook pins to the latest: **`metadata_20260707_141136`**.
- *(fill in after running the cell above)* Do `dq_reports` / `profiling_reports` share the same `metadata_` prefix, or use `raw`/`staging`/`mart` as the docstring implies? This determines the run_id filter used in Notebooks 02 and 03.

## 3. Dataset inventory

Full listing of every dataset registered in the latest `metadata_datasets` run.

In [7]:
# Dataset inventory — pinned to the latest metadata generation run only
datasets_df = conn.execute("""
    SELECT *
    FROM audit.metadata_datasets
    WHERE run_id = ?
    ORDER BY dataset_name
""", [latest_run_id]).pl()

datasets_df

metadata_id,run_id,dataset_name,file_path,file_format,file_size_bytes,row_count,column_count,source,last_modified,created_at,generated_at,generator_version,error
i64,str,str,str,str,i64,i64,i64,str,datetime[μs],datetime[μs],datetime[μs],str,str
370,"""metadata_20260707_141136""","""All_countries_holidays""","""All_countries_holidays.parquet""","""parquet""",280188,88537,10,null,2026-07-05 19:51:57.240168,2026-07-05 19:51:57.124631,2026-07-07 19:41:36.482423,"""1.1.0""",null
371,"""metadata_20260707_141136""","""CMO-Historical-Data-Monthly__Monthly_Indices""","""CMO-Historical-Data-Monthly/Monthly_Indices.parquet""","""parquet""",60540,798,17,null,2026-07-06 08:11:00.720851,2026-07-06 08:11:00.716552,2026-07-07 19:41:36.482423,"""1.1.0""",null
372,"""metadata_20260707_141136""","""CMO-Historical-Data-Monthly__Monthly_Prices""","""CMO-Historical-Data-Monthly/Monthly_Prices.parquet""","""parquet""",215298,798,72,null,2026-07-06 08:11:00.501092,2026-07-06 08:11:00.494206,2026-07-07 19:41:36.482423,"""1.1.0""",null
378,"""metadata_20260707_141136""","""FAOSTAT_A-S_E__ASTI_Expenditures_E_All_Data_(Normalized)__ASTI_Expenditures_E_All_Data_(Normalized)""","""FAOSTAT_A-S_E/ASTI_Expenditures_E_All_Data_(Normalized)/ASTI_Expenditures_E_All_Data_(Normalized).parquet""","""parquet""",50341,7789,15,null,2026-07-05 09:22:01.877702,2026-07-05 09:13:39.875607,2026-07-07 19:41:36.482423,"""1.1.0""",null
379,"""metadata_20260707_141136""","""FAOSTAT_A-S_E__ASTI_Expenditures_E_All_Data_(Normalized)__ASTI_Expenditures_E_AreaCodes""","""FAOSTAT_A-S_E/ASTI_Expenditures_E_All_Data_(Normalized)/ASTI_Expenditures_E_AreaCodes.parquet""","""parquet""",3228,163,3,null,2026-07-05 09:22:01.881710,2026-07-05 09:13:39.912897,2026-07-07 19:41:36.482423,"""1.1.0""",null
380,"""metadata_20260707_141136""","""FAOSTAT_A-S_E__ASTI_Expenditures_E_All_Data_(Normalized)__ASTI_Expenditures_E_CostCategorys""","""FAOSTAT_A-S_E/ASTI_Expenditures_E_All_Data_(Normalized)/ASTI_Expenditures_E_CostCategorys.parquet""","""parquet""",928,1,2,null,2026-07-05 09:22:01.884709,2026-07-05 09:13:39.924907,2026-07-07 19:41:36.482423,"""1.1.0""",null
381,"""metadata_20260707_141136""","""FAOSTAT_A-S_E__ASTI_Expenditures_E_All_Data_(Normalized)__ASTI_Expenditures_E_Flags""","""FAOSTAT_A-S_E/ASTI_Expenditures_E_All_Data_(Normalized)/ASTI_Expenditures_E_Flags.parquet""","""parquet""",1080,5,2,null,2026-07-05 09:22:01.886711,2026-07-05 09:13:39.935904,2026-07-07 19:41:36.482423,"""1.1.0""",null
382,"""metadata_20260707_141136""","""FAOSTAT_A-S_E__ASTI_Expenditures_E_All_Data_(Normalized)__ASTI_Expenditures_E_Indicators""","""FAOSTAT_A-S_E/ASTI_Expenditures_E_All_Data_(Normalized)/ASTI_Expenditures_E_Indicators.parquet""","""parquet""",1481,5,2,null,2026-07-05 09:22:01.888709,2026-07-05 09:13:39.949931,2026-07-07 19:41:36.482423,"""1.1.0""",null
383,"""metadata_20260707_141136""","""FAOSTAT_A-S_E__ASTI_Expenditures_E_All_Data_(Normalized)__ASTI_Expenditures_E_Institutions""","""FAOSTAT_A-S_E/ASTI_Expenditures_E_All_Data_(Normalized)/ASTI_Expenditures_E_Institutions.parquet""","""parquet""",1325,1,2,null,2026-07-05 09:22:01.891710,2026-07-05 09:13:39.968678,2026-07-07 19:41:36.482423,"""1.1.0""",null


In [8]:
# Should equal the number of datasets registered in this run
print(f"Datasets in latest run: {datasets_df.height}")

Datasets in latest run: 369


In [9]:
# Flag any datasets that failed metadata generation (error column populated)
# or have suspicious values (zero rows, zero columns, missing file_path)
issues_df = datasets_df.filter(
    (pl.col("error").is_not_null())
    | (pl.col("row_count") == 0)
    | (pl.col("column_count") == 0)
    | (pl.col("file_path").is_null())
)

print(f"Datasets with potential issues: {issues_df.height}")
issues_df

Datasets with potential issues: 19


metadata_id,run_id,dataset_name,file_path,file_format,file_size_bytes,row_count,column_count,source,last_modified,created_at,generated_at,generator_version,error
i64,str,str,str,str,i64,i64,i64,str,datetime[μs],datetime[μs],datetime[μs],str,str
400,"""metadata_20260707_141136""","""FAOSTAT_A-S_E__Climate_change_Emissions_indicators_E_All_Data_(Normalized)__Climate_change_Emissions_indicators_E_ItemCo…","""FAOSTAT_A-S_E/Climate_change_Emissions_indicators_E_All_Data_(Normalized)/Climate_change_Emissions_indicators_E_ItemCode…","""parquet""",420,0,3,null,2026-07-05 09:22:02.215942,2026-07-05 09:13:40.696053,2026-07-07 19:41:36.482423,"""1.1.0""",null
423,"""metadata_20260707_141136""","""FAOSTAT_A-S_E__Cost_Affordability_Healthy_Diet_(CoAHD)_E_All_Data_(Normalized)__Cost_Affordability_Healthy_Diet_(CoAHD)_…","""FAOSTAT_A-S_E/Cost_Affordability_Healthy_Diet_(CoAHD)_E_All_Data_(Normalized)/Cost_Affordability_Healthy_Diet_(CoAHD)_E_…","""parquet""",420,0,3,null,2026-07-05 09:22:04.625884,2026-07-05 09:13:41.872538,2026-07-07 19:41:36.482423,"""1.1.0""",null
429,"""metadata_20260707_141136""","""FAOSTAT_A-S_E__Deflators_E_All_Data_(Normalized)__Deflators_E_ItemCodes""","""FAOSTAT_A-S_E/Deflators_E_All_Data_(Normalized)/Deflators_E_ItemCodes.parquet""","""parquet""",420,0,3,null,2026-07-05 09:22:04.749232,2026-07-05 09:13:42.095009,2026-07-07 19:41:36.482423,"""1.1.0""",null
433,"""metadata_20260707_141136""","""FAOSTAT_A-S_E__Development_Assistance_to_Agriculture_E_All_Data_(Normalized)__Development_Assistance_to_Agriculture_E_It…","""FAOSTAT_A-S_E/Development_Assistance_to_Agriculture_E_All_Data_(Normalized)/Development_Assistance_to_Agriculture_E_Item…","""parquet""",420,0,3,null,2026-07-05 09:22:10.233996,2026-07-05 09:13:50.114104,2026-07-07 19:41:36.482423,"""1.1.0""",null
439,"""metadata_20260707_141136""","""FAOSTAT_A-S_E__Emissions_Agriculture_Energy_E_All_Data_(Normalized)__Emissions_Agriculture_Energy_E_ItemCodes""","""FAOSTAT_A-S_E/Emissions_Agriculture_Energy_E_All_Data_(Normalized)/Emissions_Agriculture_Energy_E_ItemCodes.parquet""","""parquet""",420,0,3,null,2026-07-05 09:22:10.328578,2026-07-05 09:13:50.354774,2026-07-07 19:41:36.482423,"""1.1.0""",null
450,"""metadata_20260707_141136""","""FAOSTAT_A-S_E__Emissions_Drained_Organic_Soils_E_All_Data_(Normalized)__Emissions_Drained_Organic_Soils_E_ItemCodes""","""FAOSTAT_A-S_E/Emissions_Drained_Organic_Soils_E_All_Data_(Normalized)/Emissions_Drained_Organic_Soils_E_ItemCodes.parque…","""parquet""",420,0,3,null,2026-07-05 09:22:10.765229,2026-07-05 09:13:51.133611,2026-07-07 19:41:36.482423,"""1.1.0""",null
456,"""metadata_20260707_141136""","""FAOSTAT_A-S_E__Emissions_Land_Use_Fires_E_All_Data_(Normalized)__Emissions_Land_Use_Fires_E_ItemCodes""","""FAOSTAT_A-S_E/Emissions_Land_Use_Fires_E_All_Data_(Normalized)/Emissions_Land_Use_Fires_E_ItemCodes.parquet""","""parquet""",420,0,3,null,2026-07-05 09:22:10.968771,2026-07-05 09:13:51.496163,2026-07-07 19:41:36.482423,"""1.1.0""",null
462,"""metadata_20260707_141136""","""FAOSTAT_A-S_E__Emissions_Land_Use_Forests_E_All_Data_(Normalized)__Emissions_Land_Use_Forests_E_ItemCodes""","""FAOSTAT_A-S_E/Emissions_Land_Use_Forests_E_All_Data_(Normalized)/Emissions_Land_Use_Forests_E_ItemCodes.parquet""","""parquet""",420,0,3,null,2026-07-05 09:22:11.053764,2026-07-05 09:13:51.686015,2026-07-07 19:41:36.482423,"""1.1.0""",null
520,"""metadata_20260707_141136""","""FAOSTAT_A-S_E__Environment_LandCover_E_All_Data_(Normalized)__Environment_LandCover_E_ItemCodes""","""FAOSTAT_A-S_E/Environment_LandCover_E_All_Data_(Normalized)/Environment_LandCover_E_ItemCodes.parquet""","""parquet""",420,0,3,null,2026-07-05 09:22:16.032836,2026-07-05 09:13:58.655413,2026-07-07 19:41:36.482423,"""1.1.0""",null


### Notes — Dataset Inventory

- **Missing datasets:** none apparent — 400+ FAOSTAT domains registered, no obvious gaps versus expected FAOSTAT bulk download coverage. *(revisit once NASA POWER / World Bank / EM-DAT sources are cross-checked — this run may be FAOSTAT-only.)*
- **Datasets with issues:** 19 datasets flagged, all following the same pattern — FAOSTAT `*_ItemCodes.parquet` auxiliary/lookup files (item code reference tables) with `row_count = 0`, `column_count = 3`, `source = null`, and file sizes of ~417–420 bytes. No `error` value recorded for any of them.
  - Confirmed via direct file read: these parquet files are genuinely empty at the source, not a metadata-generation bug.
  - These are **not core datasets** — they're FAOSTAT's item-code lookup tables, which are irrelevant to this pipeline if item codes are already resolved via the main normalized files' descriptive columns.
  - **Decision:** exclude all `*_ItemCodes.parquet` files from downstream loading/dbt staging — they carry no data and aren't needed for joins.
  - `source = null` across the board suggests the `source` field isn't being populated at all for this run — worth checking the metadata generator, since this affects every dataset, not just the 19 flagged ones.
- **Timestamps:** `generated_at` is identical (`2026-07-07 19:41:36`) across all rows — expected, since that's the batch-run timestamp. `last_modified` / `created_at` cluster around `2026-07-05`, consistent with when the raw files were originally downloaded. Nothing suspiciously stale relative to that.

## 4. Column-level metadata

Full column inventory for the latest run — used to spot join keys,
inconsistent naming (e.g. `country` vs `country_name`), and datatype mismatches across datasets.

In [10]:
# Column-level metadata — pinned to the same latest run
columns_df = conn.execute("""
    SELECT *
    FROM audit.metadata_columns
    WHERE run_id = ?
    ORDER BY dataset_name, column_name
""", [latest_run_id]).pl()

columns_df.head(20)

column_id,run_id,dataset_name,column_name,data_type,nullable,null_count,null_percent,is_numeric,is_temporal,is_categorical,sample_values_json
i64,str,str,str,str,bool,i64,f64,bool,bool,bool,str
1987,"""metadata_20260707_141136""","""All_countries_holidays""","""counties""","""Null""",true,88537,100.0,false,false,true,"""[]"""
1980,"""metadata_20260707_141136""","""All_countries_holidays""","""country_code""","""String""",false,0,0.0,false,false,true,"""[""AD"", ""AE"", ""AF""]"""
1981,"""metadata_20260707_141136""","""All_countries_holidays""","""country_name""","""String""",false,0,0.0,false,false,true,"""[""Andorra"", ""United Arab Emirates"", ""Afghanistan""]"""
1982,"""metadata_20260707_141136""","""All_countries_holidays""","""date""","""String""",false,0,0.0,false,false,true,"""[""2000-01-01"", ""2000-01-06"", ""2000-03-06""]"""
1985,"""metadata_20260707_141136""","""All_countries_holidays""","""fixed""","""Null""",true,88537,100.0,false,false,true,"""[]"""
1986,"""metadata_20260707_141136""","""All_countries_holidays""","""global""","""Null""",true,88537,100.0,false,false,true,"""[]"""
1988,"""metadata_20260707_141136""","""All_countries_holidays""","""launch_year""","""Null""",true,88537,100.0,false,false,true,"""[]"""
1983,"""metadata_20260707_141136""","""All_countries_holidays""","""local_name""","""String""",false,0,0.0,false,false,true,"""[""New Year's Day"", ""Epiphany"", ""Carnival""]"""
1984,"""metadata_20260707_141136""","""All_countries_holidays""","""name""","""String""",false,0,0.0,false,false,true,"""[""New Year's Day"", ""Epiphany"", ""Carnival""]"""


## 5. Cross-check: column counts per dataset

Sanity check — does the column count in `metadata_columns` match what's
recorded in `metadata_datasets.column_count`? Mismatches indicate stale or
inconsistent metadata generation.

In [11]:
# Column count per dataset, computed directly from metadata_columns
computed_counts = (
    columns_df
    .group_by("dataset_name")
    .agg(pl.count("column_name").alias("computed_column_count"))
    .sort("dataset_name")
)

computed_counts

dataset_name,computed_column_count
str,u32
"""All_countries_holidays""",10
"""CMO-Historical-Data-Monthly__Monthly_Indices""",17
"""CMO-Historical-Data-Monthly__Monthly_Prices""",72
"""FAOSTAT_A-S_E__ASTI_Expenditures_E_All_Data_(Normalized)__ASTI_Expenditures_E_All_Data_(Normalized)""",15
"""FAOSTAT_A-S_E__ASTI_Expenditures_E_All_Data_(Normalized)__ASTI_Expenditures_E_AreaCodes""",3
"""FAOSTAT_A-S_E__ASTI_Expenditures_E_All_Data_(Normalized)__ASTI_Expenditures_E_CostCategorys""",2
"""FAOSTAT_A-S_E__ASTI_Expenditures_E_All_Data_(Normalized)__ASTI_Expenditures_E_Flags""",2
"""FAOSTAT_A-S_E__ASTI_Expenditures_E_All_Data_(Normalized)__ASTI_Expenditures_E_Indicators""",2
"""FAOSTAT_A-S_E__ASTI_Expenditures_E_All_Data_(Normalized)__ASTI_Expenditures_E_Institutions""",2


In [12]:
# Join against metadata_datasets' own recorded column_count to catch mismatches
mismatch_check = (
    datasets_df
    .select(["dataset_name", "column_count"])
    .rename({"column_count": "recorded_column_count"})
    .join(computed_counts, on="dataset_name", how="left")
    .filter(
        pl.col("recorded_column_count") != pl.col("computed_column_count")
    )
)

print(f"Datasets with mismatched column counts: {mismatch_check.height}")
mismatch_check  # should be empty if metadata is in sync

Datasets with mismatched column counts: 0


dataset_name,recorded_column_count,computed_column_count
str,i64,u32


### Notes — Column Metadata

- *(fill in after reviewing output)*
- Any naming inconsistencies across datasets for the same real-world concept (e.g. `country` vs `country_code` vs `area`)?
- Any datasets with unexpectedly few columns?
- Any mismatches surfaced above — and if so, likely cause (partial write, schema drift, generator bug)?

In [16]:
# Close the connection
conn.close()
print("Connection closed")

Connection closed
